In [ ]:
!pip install pandapower matplotlib seaborn numpy pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.9/292.9 kB 23.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [ ]:
import pandapower as pp
import pandapower.networks as pn
import pandapower.plotting as plot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f'pandapower version: {pp.__version__}')
print('All imports successful.')

pandapower version: 3.4.0
All imports successful.


In [ ]:
def label_stability(vm_series):
    total_buses = len(vm_series)
    buses_below = (vm_series < 0.90).sum()
    pct_below = buses_below / total_buses

    if pct_below > 0.20:
        return "UNSTABLE"
    elif 0.05 < pct_below <= 0.20:
        return "WARNING"
    else:
        return "STABLE"

In [ ]:
results = []
net = pn.case39()
base_p = net.load['p_mw'].values.copy()
base_q = net.load['q_mvar'].values.copy()
load_factors = np.arange(0.60, 1.25, 0.05)
line_indices = list(range(len(net.line))) + [-1]  # -1 means no outage

for lf in load_factors:
    for line_idx in line_indices:
        net_temp = pn.case39()
        net_temp.f_hz = 50.0
        net_temp.load['p_mw'] = base_p * lf
        net_temp.load['q_mvar'] = base_q * lf

        if line_idx != -1:
            net_temp.line.at[line_idx, 'in_service'] = False

        try:
            pp.runpp(net_temp, algorithm='nr', calculate_voltage_angles=True, max_iteration=50)
            if net_temp.converged:
                vm_series = net_temp.res_bus['vm_pu']
                stability_label = label_stability(vm_series)

                for bus_id in net_temp.res_bus.index:
                    vm = net_temp.res_bus.loc[bus_id, 'vm_pu']
                    va = net_temp.res_bus.loc[bus_id, 'va_degree']
                    results.append({
                        'Bus ID': bus_id,
                        'Voltage Magnitude': vm,
                        'Voltage Angle': va,
                        'freq_hz': 50.0 + np.random.normal(0, 0.01),
                        'Load Factor': lf,
                        'Line Outage': line_idx,
                        'Stability Label': stability_label
                    })
        except:
            pass

df = pd.DataFrame(results)
print(df.shape)
print(df['Stability Label'].value_counts())
print(df.groupby('Stability Label')['Load Factor'].agg(['min', 'max']))

(18213, 7)
Stability Label
STABLE      16341
WARNING      1443
UNSTABLE      429
Name: count, dtype: int64
                 min  max
Stability Label          
STABLE           0.6  1.2
UNSTABLE         0.6  1.2
WARNING          0.6  1.2


In [ ]:
df.to_csv('voltage_stability_dataset.csv', index=False)
print('Dataset saved.')
print(df.head())

Dataset saved.
   Bus ID  Voltage Magnitude  Voltage Angle    freq_hz  Load Factor  \
0       0           1.030957      40.330800  50.002858          0.6   
1       1           1.014978      58.253833  50.009634          0.6   
2       2           0.971241      52.192754  49.991081          0.6   
3       3           0.922611      40.375760  49.978707          0.6   
4       4           0.915772      32.559313  50.001585          0.6   

   Line Outage Stability Label  
0            0          STABLE  
1            0          STABLE  
2            0          STABLE  
3            0          STABLE  
4            0          STABLE  


In [ ]:
from google.colab import files
files.download('voltage_stability_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>